# **Day 74: Time Zones**

On Day 32 we used dates and times **without** time zones. This becomes a problem when we work with data from other countries or from **satellites**.

**Very important for Remote Sensing:** Satellite image times are always written in **UTC** (Coordinated Universal Time, the world's standard time). India's time (IST) is **UTC + 5 hours 30 minutes**.

On Windows, we may need to run `pip install tzdata` once, so that Python knows the time zone rules.

In [1]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

IST = ZoneInfo("Asia/Kolkata")
print(IST)

naive = datetime(2024, 10, 10, 18, 30)                   # No time zone
aware = datetime(2024, 10, 10, 18, 30, tzinfo=IST)       # With time zone
print(naive, "| time zone:", naive.tzinfo)
print(aware, "| time zone:", aware.tzinfo, "| offset:", aware.utcoffset())

Asia/Kolkata
2024-10-10 18:30:00 | time zone: None
2024-10-10 18:30:00+05:30 | time zone: Asia/Kolkata | offset: 5:30:00


**Fun fact:** The official name of India's time zone is **"Asia/Kolkata"**. The time zone of the whole country is named after Kolkata!

## **Example from Remote Sensing: When Was the Image Really Taken?**
The Landsat metadata file (Day 34) said: `SCENE_CENTER_TIME = "04:36:12.5Z"`. The **Z** means UTC. What was the time in Kolkata?

In [2]:
utc_time = datetime(2023, 11, 10, 4, 36, 12, tzinfo=timezone.utc)
local_time = utc_time.astimezone(IST)

print("Image time in UTC :", utc_time.strftime("%d %b %Y, %I:%M %p"))
print("Image time in IST :", local_time.strftime("%d %b %Y, %I:%M %p"))

Image time in UTC : 10 Nov 2023, 04:36 AM
Image time in IST : 10 Nov 2023, 10:06 AM


The image was taken at about **10:06 in the morning** in Kolkata. Landsat and Sentinel-2 satellites pass over India in the **late morning**, when the sunlight is good and there are usually fewer clouds than in the afternoon.

In [3]:
# Sentinel-2 file names also have UTC time: S2A_MSIL2A_20240105T045141_...
name = "S2A_MSIL2A_20240105T045141_N0510_R076_T45QXF_20240105T083015"
utc = datetime.strptime(name.split("_")[2], "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)
print("Sentinel-2 image taken at", utc.astimezone(IST).strftime("%I:%M %p IST on %d %B %Y"))

Sentinel-2 image taken at 10:21 AM IST on 05 January 2024


## **Converting Between Time Zones**

In [4]:
zones = {
    "Kolkata": "Asia/Kolkata",
    "London": "Europe/London",
    "New York": "America/New_York",
    "Dubai": "Asia/Dubai",
    "Kathmandu": "Asia/Kathmandu",
    "Dhaka": "Asia/Dhaka",
}
video_call = datetime(2024, 10, 11, 20, 0, tzinfo=IST)      # Ashtami evening family call

print("A video call at 8 PM Kolkata time is:")
for city, tz in zones.items():
    t = video_call.astimezone(ZoneInfo(tz))
    print(f"  {city:<10}{t:%d %b %I:%M %p}  (UTC{t.strftime('%z')})")

A video call at 8 PM Kolkata time is:
  Kolkata   11 Oct 08:00 PM  (UTC+0530)
  London    11 Oct 03:30 PM  (UTC+0100)
  New York  11 Oct 10:30 AM  (UTC-0400)
  Dubai     11 Oct 06:30 PM  (UTC+0400)
  Kathmandu 11 Oct 08:15 PM  (UTC+0545)
  Dhaka     11 Oct 08:30 PM  (UTC+0600)


Nepal (Kathmandu) is at UTC **+5:45**. Time zones do not always differ by whole hours.

## **Rule: Store Time in UTC, Show It in Local Time**
This is the standard practice everywhere:
1. **Save** all times in UTC (so there is no confusion).
2. **Convert** to local time only when showing it to people.

In [5]:
saved = datetime(2024, 10, 11, 14, 30, tzinfo=timezone.utc)
print("Saved (UTC):", saved.isoformat())
print("Shown (IST):", saved.astimezone(IST).isoformat())

# Comparing a time with a time zone and one without gives an error
# print(naive < aware)     # This will raise a TypeError

Saved (UTC): 2024-10-11T14:30:00+00:00
Shown (IST): 2024-10-11T20:00:00+05:30


## **The time Module**
- `time.time()`: seconds since 1 January 1970 (UTC). This is called the **Unix timestamp**.
- `time.perf_counter()`: the best clock for measuring **how long** something takes.

In [6]:
import time
print(time.strftime("%Y-%m-%d", time.gmtime(1_700_000_000)))

start = time.perf_counter()
sum(range(1_000_000))
print(f"Took {(time.perf_counter() - start) * 1000:.2f} milliseconds")

2023-11-14
Took 9.16 milliseconds


## **Practice Problem: Night Train Across Midnight**

In [7]:
departure = datetime(2024, 10, 12, 22, 5, tzinfo=IST)
arrival = datetime(2024, 10, 13, 8, 0, tzinfo=IST)
duration = arrival - departure
hours, remainder = divmod(duration.seconds, 3600)
print(f"Journey time: {hours} hours {remainder // 60} minutes")
print("Departure in UTC:", departure.astimezone(timezone.utc).strftime("%d %b %H:%M"))

Journey time: 9 hours 55 minutes
Departure in UTC: 12 Oct 16:35


In [8]:
# Quick question
# A satellite image time is 05:00 UTC. What is the time in IST?